# 02: Veri Dönüşümü ve Dengeleme: Encoding, Ölçekleme, Feature Engineering ve SMOTE

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini

**Girdi:** Kişi 1'in `01_data_preprocessing` notebook'unda hazırladığı `train.csv` ve `test.csv`. Bu dosyalarda sızıntılı sütunlar çıkarılmış, imkânsız değerler temizlenmiş ve eksik değerler (yalnızca eğitim verisinin istatistikleriyle) doldurulmuştur.

**Bu notebook'ta yapılanlar:**
1. Encoding: sayıya çevrilmesi gereken sütunlar
2. Ölçekleme (standardizasyon)
3. Feature engineering: tıbben anlamlı yeni özellikler
4. Adımların veri üzerindeki etkisi
5. Veri dengeleme: SMOTE
6. Ekibin ortak kullanacağı Pipeline

Adımların kodu ekibin ortak kullandığı **`src/preprocessing.py`** dosyasındadır; ortak sabitler (`RANDOM_STATE` vb.) `src/config.py`'dedir.

**Colab'da çalıştırmak için:** Sol menüdeki klasör simgesinden `train.csv`, `test.csv` (Kişi 1'in çıktıları), `config.py` ve `preprocessing.py` dosyalarını yükleyin.

```
Kişi 1'in verisi → ① feature engineering → ② encoding + ölçekleme → ③ dengeleme (SMOTE) → ④ model
```

> **Veri sızıntısı önlemi:** Tüm adımlar tek bir Pipeline içinde olduğu için ölçekleme parametreleri ve SMOTE yalnızca **eğitim verisinden** öğrenilir. Cross validation sırasında her katta yalnızca o katın eğitim kısmına uygulanır. Test seti bu notebook'ta kullanılmaz.

In [ ]:
# imbalanced-learn ships with Colab; locally: %pip install -q imbalanced-learn
import sys
import warnings
from pathlib import Path

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import SMOTE

from config import RANDOM_STATE
from preprocessing import FeatureEngineer, build_pipeline, build_preprocessor, load_data

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

X_train, X_test, y_train, y_test = load_data()
print("Eğitim verisi:", X_train.shape, "| inme oranı: %.2f%%" % (y_train.mean() * 100))
print("Eksik değer:", X_train.isna().sum().sum())
X_train.head()

## 1. Encoding

Framingham verisinde sütunların çoğu zaten sayısal olarak kodlanmıştır. Yine de bazı sütunların kodlaması model için yanıltıcıdır:

| Sütun | Ham kodlama | Yapılan işlem | Gerekçe |
|---|---|---|---|
| `SEX` | 1 = erkek, 2 = kadın | `SEX_female` (0 = erkek, 1 = kadın) | 1/2 kodlaması modele "kadın, erkeğin iki katı" gibi anlamsız bir sayısal ilişki verir. 0/1 kodlama ikili bir değişken için doğru gösterimdir. |
| `educ` | 1–4 (eğitim düzeyi) | Sayısal olarak tutuldu, ölçeklendi | Eğitim düzeyleri arasında doğal bir **sıra** vardır (sıralı/ordinal değişken), bu yüzden sayısal değer anlamlıdır. |
| `bp_stage`, `bmi_category` (feature engineering ile eklenen) | Metin kategoriler | One-hot encoding | Model metin okuyamaz. Her kategori ayrı bir 0/1 sütuna dönüştürülür. |
| `CURSMOKE`, `DIABETES`, `BPMEDS`, `PREVCHD`, `PREVAP`, `PREVMI`, `PREVHYP` | Zaten 0/1 | Değiştirilmedi | Zaten doğru biçimde |

## 2. Ölçekleme (standardizasyon)

Sürekli değişkenlerin değer aralıkları çok farklıdır: örneğin `AGE` 32–70, `TOTCHOL` 100–600, `SYSBP` 80–300. Logistic Regression, SVM ve KNN gibi katsayı ya da mesafe tabanlı modellerde büyük değerli sütunlar baskın çıkar. Bu yüzden sürekli değişkenlere **StandardScaler** uygulanır: her sütun ortalaması 0, standart sapması 1 olacak şekilde dönüştürülür.

Ağaç tabanlı modeller (Random Forest, XGBoost) ölçeklemeden etkilenmez. Ekip tek bir ortak Pipeline kullandığı için ölçekleme onlarda da uygulanır ama sonuçlarını değiştirmez.

## 3. Feature engineering

Mevcut ölçümlerden tıbben anlamlı 5 yeni özellik türetildi. Eşikler sabit tıbbi değerlerdir, veriden öğrenilmez:

| Özellik | Tanım | Tıbbi gerekçe |
|---|---|---|
| `pulse_pressure` | Nabız basıncı = SYSBP − DIABP | Yüksek nabız basıncı damar sertliğinin göstergesidir ve inme riskiyle ilişkilidir. |
| `mean_arterial_pressure` | Ortalama arter basıncı = DIABP + (SYSBP − DIABP) / 3 | Beyin dahil organlara ulaşan ortalama kan basıncı |
| `bp_stage` | ACC/AHA 2017 tansiyon evresi: `normal` (<120/<80) · `elevated` yüksek normal (120–129/<80) · `stage1` evre 1 (130–139 veya 80–89) · `stage2` evre 2 (≥140 veya ≥90) | Hipertansiyon, inmenin en güçlü değiştirilebilir risk faktörüdür. Evreler, riskin eşiklerden sonra arttığı klinik bilgisini modele verir. |
| `bmi_category` | WHO: `underweight` zayıf (<18.5) · `normal` (<25) · `overweight` fazla kilolu (<30) · `obese` obez (30+) | Obezite bilinen bir kardiyovasküler risk faktörüdür. |
| `comorbidity_count` | DIABETES + PREVHYP + PREVCHD (0–3) | Eşlik eden hastalık sayısı arttıkça risk birikir. |

## 4. Adımların veri üzerindeki etkisi

### ① Feature engineering sonrası (yeni sütunlar)

In [ ]:
fe = FeatureEngineer().fit(X_train)
X_fe = fe.transform(X_train)
X_fe[["SYSBP", "DIABP", "BMI", "DIABETES", "PREVHYP", "PREVCHD",
      "pulse_pressure", "mean_arterial_pressure", "bp_stage", "bmi_category", "comorbidity_count"]].head()

In [ ]:
for col in ["bp_stage", "bmi_category"]:
    table = pd.DataFrame({
        "count": X_fe[col].value_counts(),
        "stroke_rate_%": (y_train.groupby(X_fe[col]).mean() * 100).round(1),
    })
    print(f"--- {col} ---\n{table}\n")

### ② Encoding + ölçekleme sonrası: modele giren veri

In [ ]:
preprocessor = build_preprocessor(feature_engineering=True).fit(X_train)
X_train_ready = preprocessor.transform(X_train)

print(f"Kişi 1'in verisi: {X_train.shape[1]} sütun  →  modele giren veri: {X_train_ready.shape[1]} sütun")
print("Eksik değer kaldı mı?", X_train_ready.isna().any().any())
X_train_ready.head()

In [ ]:
# Scaled columns should have mean ≈ 0 and std ≈ 1
scaled_cols = ["AGE", "TOTCHOL", "SYSBP", "DIABP", "BMI", "GLUCOSE", "pulse_pressure"]
X_train_ready[scaled_cols].describe().loc[["mean", "std"]].round(3)

## 5. Veri dengeleme: SMOTE

Eğitim verisinde inme geçiren kişiler azınlıktadır. Dengesiz veride model azınlık sınıfını (inme) yeterince öğrenemez ve çoğu kişiye "inme yok" diyerek yüksek doğruluk elde eder. Bu durum, inme riskini önceden yakalamayı amaçlayan bir tarama modeli için kabul edilemez.

**SMOTE (Synthetic Minority Over-sampling Technique)**, azınlık sınıfındaki her örneğin en yakın komşularını bulur ve aralarında yeni **sentetik** örnekler üretir. Böylece eğitim verisindeki iki sınıf eşit sayıya gelir.

- SMOTE mesafe hesabı yaptığı için **ölçeklemeden sonra** uygulanır.
- SMOTE **yalnızca eğitim verisine** uygulanır. Test verisine uygulansaydı model gerçek olmayan (sentetik) hastalarla değerlendirilmiş olurdu ve sonuçlar gerçeği yansıtmazdı.

In [ ]:
X_balanced, y_balanced = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train_ready, y_train)

balance = pd.DataFrame({
    "before_smote": y_train.value_counts().sort_index(),
    "after_smote": y_balanced.value_counts().sort_index(),
}).rename(index={0: "no_stroke (0)", 1: "stroke (1)"})
balance.loc["total"] = balance.sum()
balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for ax, (title, y) in zip(axes, [("SMOTE öncesi", y_train), ("SMOTE sonrası", y_balanced)]):
    counts = y.value_counts().sort_index()
    ax.bar(["inme yok", "inme var"], counts.values, color=["#4C72B0", "#DD8452"])
    for i, v in enumerate(counts.values):
        ax.text(i, v, str(v), ha="center", va="bottom")
    ax.set_title(title)
axes[0].set_ylabel("kişi sayısı (eğitim verisi)")
plt.tight_layout()
plt.savefig("02_smote_before_after.png", dpi=200)  # figure for the report
plt.show()

> **Not:** Yukarıdaki hücre SMOTE'un etkisini göstermek içindir. Modeller eğitilirken SMOTE elle uygulanmaz; `build_pipeline()` içinde otomatik olarak, cross validation'ın her katında yalnızca eğitim kısmına uygulanır.

## 6. Ekip için kullanım

Her ekip üyesi modelini aşağıdaki şekilde kurar. Kişi 1'in verisi olduğu gibi verilir; feature engineering, encoding, ölçekleme ve dengeleme Pipeline'ın içinde yapılır.

```python
from preprocessing import build_pipeline, load_data

X_train, X_test, y_train, y_test = load_data()   # data/ (repo) veya çalışma klasörü (Colab)
pipe = build_pipeline(model, balancing="smote")   # "smote" | "class_weight" | "none"
pipe.fit(X_train, y_train)
```

Colab'da `config.py`, `preprocessing.py`, `train.csv` ve `test.csv` dosyalarını sol menüden yüklemek yeterlidir (değerlendirme ve XAI için ayrıca `metrics.py` ve `explain.py`).

- **XAI ile birlikte:** Eğitilen `pipe`, Kişi 4'ün `explain(pipe, X_train, X_test, model_name=...)` fonksiyonuna (`src/explain.py`) doğrudan verilebilir. SHAP grafiklerinde yeni sütun isimleri (`SEX_female`, `bp_stage_stage2` vb.) görünür.
- GridSearch'te model parametreleri `model__` ön ekiyle yazılır, ör. `{"model__C": [0.1, 1, 10]}`.
- `build_pipeline` verilen modelin bir kopyasını (`clone`) kullanır; aynı model nesnesi farklı dengeleme seçenekleriyle tekrar kullanılabilir.
- **Dengeleme:** Varsayılan SMOTE'tur. Ağaç modellerinde (Random Forest, XGBoost) SMOTE'un sentetik örnekleri ezberlenebildiği için `class_weight` da denenmeli ve hangisinin daha iyi olduğu karşılaştırılmalıdır.
- **XGBoost:** `class_weight` parametresi olmadığı için karşılığı kullanılır: `XGBClassifier(scale_pos_weight=scale_pos_weight(y_train))` ve `balancing="none"`.

In [ ]:
# End-to-end check on the training data only
from sklearn.linear_model import LogisticRegression

pipe = build_pipeline(LogisticRegression(max_iter=3000, random_state=RANDOM_STATE), balancing="smote")
pipe.fit(X_train, y_train)
print("Pipeline adımları:", [name for name, _ in pipe.steps])
print("İlk 5 kişinin inme olasılığı:", pipe.predict_proba(X_train.head())[:, 1].round(3))

## 7. Özet (rapor için)

- Framingham verisinin çoğu sütunu zaten sayısal olduğu için encoding sınırlı kaldı: `SEX` 1/2'den 0/1'e (`SEX_female`) çevrildi, `educ` sıralı olduğu için sayısal tutuldu, yeni kategorik özellikler one-hot ile kodlandı.
- Sürekli değişkenler StandardScaler ile standartlaştırıldı (ortalama 0, standart sapma 1).
- Tansiyon, kilo ve eşlik eden hastalık bilgilerinden 5 tıbbi özellik türetildi: nabız basıncı, ortalama arter basıncı, ACC/AHA tansiyon evresi, WHO BMI sınıfı, eşlik eden hastalık sayısı.
- Sınıf dengesizliği SMOTE ile giderildi; SMOTE yalnızca eğitim verisine, Pipeline içinde uygulanır.
- Tüm adımlar `build_pipeline()` fonksiyonunda (`src/preprocessing.py`) birleştirildi; ekipteki herkes modelini bu fonksiyonla kurar.